# CineEmotion Sub-Project: Master Kaggle Submission

This pipeline synthesizes:
1. **DeBERTa-V3 (LoRA) + Riemannian Spherical Routing**
2. **Gated Chronological Scene Pooling**
3. **Mamba-2 Continuous Core + Differentiable Episodic Memory**
4. **Conditional Variational Autoencoder (CVAE)**

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Local Imports
from config import pipeline_config
from models.module1_encoder import RiemannianUtteranceEncoder
from models.module2_3_narrative import SceneNarrativeEngine
from models.module4_planner import CVAEMusicPlanner
from training.trainer import OrchestratorTrainer

# Ensure hardware availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing on hardware: {device}")

In [ ]:
class CineMusicPipeline(nn.Module):
    """
    Master Network. Wraps all deep mathematical modules into a single, 
    forward-propagating directed acyclic graph.
    """
    def __init__(self, config):
        super(CineMusicPipeline, self).__init__()
        
        # Module 1: Utterance level (DeBERTa + LoRA + Spherical Math)
        self.utterance_encoder = RiemannianUtteranceEncoder(config.module1)
        dim_m1 = self.utterance_encoder.config.hidden_size # usually 1024 for deberta-large
        
        # Module 2 & 3: Narrative Engine (Scene Pooler + Mamba + Memory)
        self.narrative_engine = SceneNarrativeEngine(config.module2_3, m1_hidden_size=dim_m1)
        dim_m3 = config.module2_3.get('mamba_d_model', 128)
        
        # Module 4: Categorical / Continuous Target CVAE Predictor
        self.music_planner = CVAEMusicPlanner(config.module4, mamba_d_model=dim_m3)

    def forward(self, input_ids, attention_mask):
        """
        input_ids: [batch_size, num_scenes, num_utterances, seq_length]
        """
        bs, num_scenes, num_utts, seq_len = input_ids.shape
        
        # Flatten for Encoder
        flat_input = input_ids.view(-1, seq_len)
        flat_mask = attention_mask.view(-1, seq_len)
        
        # Pass 1: Encode Utterances
        utterance_embeddings, mod1_contrastive_loss = self.utterance_encoder(flat_input, flat_mask)
        
        # Reshape for Narrative
        batched_utterances = utterance_embeddings.view(bs, num_scenes, num_utts, -1)
        utterance_pad_mask = attention_mask.view(bs, num_scenes, num_utts, seq_len).sum(dim=-1) == 0
        
        # Pass 2: Sequence Mamba & Differentiable Memory
        narrative_states = self.narrative_engine(batched_utterances, utterance_pad_mask)
        
        # Pass 3: Predict Variations
        outputs = self.music_planner(narrative_states)
        
        # Bubble up the structural losses
        outputs['contrastive_loss'] = mod1_contrastive_loss
        return outputs

In [ ]:
# --------------------------------------------------------
# Kaggle Execution Block
# --------------------------------------------------------

# 1. Load the Pipeline Architecture
model = CineMusicPipeline(pipeline_config).to(device)

# 2. Optimizer Configuration (Avoiding Memory Overload)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=0.01)

# 3. Note for User:
# Instantiate your actual Dataset loader pointing to the JSON `datasets` folder here.
# dataloader = DataLoader(YourDataset(json_dir="datasets"), batch_size=4, shuffle=True)
# Below is dummy logic strictly so the code compiles upon Run All.
class DummyKaggleDataloader:
    def __iter__(self):
        for _ in range(3):
            yield {
                'input_ids': torch.randint(0, 1000, (2, 5, 4, 128)),
                'attention_mask': torch.ones((2, 5, 4, 128)) # Unmasked structurally for test
            }
    def __len__(self):
        return 3
        
dataloader = DummyKaggleDataloader()

# 4. Initialize Mixed-Precision Orchestrator
trainer = OrchestratorTrainer(
    model=model, 
    dataloader=dataloader, 
    optimizer=optimizer, 
    config=pipeline_config,
    device=device
)

# 5. Execute
print("Initiating Deep Mathematical Pipeline Training...")
for epoch in range(1, 4):  # Start with 3 epochs for memory testing
    trainer.train_epoch(epoch)
print("CineMusic Pipeline Pre-Training Successful!")